# 🔧 Phase 3 — Feature Engineering

> **Malicious PDF Detector** — Static PDF Feature Extraction Pipeline

This notebook demonstrates the complete feature engineering pipeline:
1. **Structural Feature Extraction** (25 features) — regex-based PDF keyword scanning
2. **Metadata Feature Extraction** (12 features) — PyPDF2-powered document inspection
3. **Feature Vectorization** — combine → normalize → persist
4. **Scaler Fitting** — fit `StandardScaler` on training data
5. **Validation** — compare extracted features with CIC ground truth

In [ ]:
# === Cell 1: Setup & Imports ===
import sys
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Ensure project root is on the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    FEATURE_COLUMNS, PROCESSED_DATA_DIR, SAMPLE_PDFS_DIR,
    MODELS_DIR, FIGURES_DIR, RESULTS_DIR, RANDOM_SEED
)
from src.features.structural import extract_structural_features
from src.features.metadata import extract_metadata_features
from src.features.vectorizer import (
    combine_features, combine_features_df, fit_scaler, transform,
    pdf_to_vector, extract_features_dict, compute_benign_baseline,
    load_scaler, save_scaler
)
from src.utils.logger import get_logger

logger = get_logger('notebook.03_feature_engineering')

print('🔧 Phase 3: Feature Engineering Pipeline')
print(f'   Project Root: {project_root}')
print(f'   Feature Count: {len(FEATURE_COLUMNS)}')
print(f'   Structural: 25 | Metadata: 12')
print(f'   Sample PDFs Dir: {SAMPLE_PDFS_DIR}')

In [ ]:
# === Cell 2: Demo Structural Extraction on Sample PDFs ===

print('='*70)
print('📐 STRUCTURAL FEATURE EXTRACTION (25 features)')
print('='*70)
print()

# Discover sample PDFs
sample_pdfs = list(SAMPLE_PDFS_DIR.glob('*.pdf'))
print(f'Found {len(sample_pdfs)} sample PDFs in {SAMPLE_PDFS_DIR}')

if not sample_pdfs:
    print('⚠️  No sample PDFs found. Creating minimal test PDFs...')
    # Create minimal test PDFs if none exist
    SAMPLE_PDFS_DIR.mkdir(parents=True, exist_ok=True)
    
    benign_pdf = b'''%PDF-1.4
1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj
2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj
3 0 obj<</Type/Page/Parent 2 0 R/MediaBox[0 0 612 792]/Contents 4 0 R/Resources<</Font<</F1 5 0 R>>>>>>endobj
4 0 obj<</Length 44>>stream
BT /F1 12 Tf 100 700 Td (Hello World) Tj ET
endstream endobj
5 0 obj<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>endobj
xref
0 6
trailer<</Size 6/Root 1 0 R>>
startxref
0
%%EOF'''
    (SAMPLE_PDFS_DIR / 'benign_sample.pdf').write_bytes(benign_pdf)
    sample_pdfs = list(SAMPLE_PDFS_DIR.glob('*.pdf'))
    print(f'  Created {len(sample_pdfs)} test PDFs')

structural_results = {}
for pdf_path in sorted(sample_pdfs):
    print(f'\n--- {pdf_path.name} ({pdf_path.stat().st_size:,} bytes) ---')
    feats = extract_structural_features(pdf_path)
    structural_results[pdf_path.name] = feats
    
    # Display non-zero features
    nonzero = {k: v for k, v in feats.items() if v > 0}
    print(f'  Non-zero features: {len(nonzero)} / 25')
    for k, v in sorted(nonzero.items(), key=lambda x: -x[1]):
        indicator = '🔴' if k in ['js_count', 'javascript_count', 'openaction_count',
                                   'launch_count', 'submitform_count', 'xfa_count',
                                   'obfuscation_count'] else '🟢'
        print(f'    {indicator} {k}: {v}')

# Create summary DataFrame
struct_df = pd.DataFrame(structural_results).T
print('\n📊 Structural Features Summary Table:')
display(struct_df.style.background_gradient(cmap='YlOrRd', axis=1)
        .format('{:.1f}'))

In [ ]:
# === Cell 3: Demo Metadata Extraction on Sample PDFs ===

print('='*70)
print('📋 METADATA FEATURE EXTRACTION (12 features)')
print('='*70)
print()

metadata_results = {}
for pdf_path in sorted(sample_pdfs):
    print(f'\n--- {pdf_path.name} ---')
    feats = extract_metadata_features(pdf_path)
    metadata_results[pdf_path.name] = feats
    
    # Display all features
    for k, v in feats.items():
        status = '✅' if v > 0 else '⬜'
        print(f'    {status} {k}: {v}')

# Create summary DataFrame
meta_df = pd.DataFrame(metadata_results).T
print('\n📊 Metadata Features Summary Table:')
display(meta_df.style.background_gradient(cmap='Blues', axis=1)
        .format('{:.1f}'))

In [ ]:
# === Cell 4: Combined Feature Vectors as DataFrame ===

print('='*70)
print('🔗 COMBINED FEATURE VECTORS (37 dimensions)')
print('='*70)
print()

combined_rows = []
for pdf_path in sorted(sample_pdfs):
    s = structural_results[pdf_path.name]
    m = metadata_results[pdf_path.name]
    vec = combine_features(s, m)
    combined_rows.append({
        'file': pdf_path.name,
        'vector_shape': vec.shape,
        'non_zero_count': int(np.count_nonzero(vec)),
        'total_sum': float(vec.sum()),
        'max_value': float(vec.max()),
    })
    print(f'{pdf_path.name}:')
    print(f'  Shape: {vec.shape}')
    print(f'  Non-zero: {np.count_nonzero(vec)} / {len(vec)}')
    print(f'  Sum: {vec.sum():.2f}')
    print(f'  Max: {vec.max():.2f}')
    print()

# Show full combined feature DataFrame
combined_dfs = []
for pdf_path in sorted(sample_pdfs):
    df_row = combine_features_df(
        structural_results[pdf_path.name],
        metadata_results[pdf_path.name]
    )
    df_row.index = [pdf_path.name]
    combined_dfs.append(df_row)

if combined_dfs:
    full_df = pd.concat(combined_dfs)
    print('Full Combined Feature Matrix:')
    display(full_df.T.style.background_gradient(cmap='viridis', axis=1)
            .format('{:.2f}'))
    print(f'\n✅ Feature column order matches config.FEATURE_COLUMNS: '
          f'{list(full_df.columns) == FEATURE_COLUMNS}')

In [ ]:
# === Cell 5: Fit Scaler on Training Data ===

print('='*70)
print('⚖️ SCALER FITTING & NORMALIZATION')
print('='*70)
print()

# Load training data if available
train_path = PROCESSED_DATA_DIR / 'train.csv'
cleaned_path = PROCESSED_DATA_DIR / 'cleaned.csv'

if train_path.exists():
    train_df = pd.read_csv(train_path)
    data_source = 'train.csv'
elif cleaned_path.exists():
    train_df = pd.read_csv(cleaned_path)
    data_source = 'cleaned.csv (fallback)'
else:
    print('⚠️  No processed data found. Using synthetic data for demo.')
    np.random.seed(RANDOM_SEED)
    train_df = pd.DataFrame(
        np.random.rand(100, len(FEATURE_COLUMNS)) * 10,
        columns=FEATURE_COLUMNS
    )
    train_df['Class'] = np.random.randint(0, 2, 100)
    data_source = 'synthetic (demo only)'

print(f'Data source: {data_source}')
print(f'Shape: {train_df.shape}')

# Extract feature columns
feature_cols = [c for c in FEATURE_COLUMNS if c in train_df.columns]
X_train = train_df[feature_cols].values

# Detect label column
label_col = 'Class' if 'Class' in train_df.columns else train_df.columns[-1]
y_train = train_df[label_col].values

print(f'Feature matrix shape: {X_train.shape}')
print(f'Label distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')

# Fit scaler
scaler = fit_scaler(X_train, save=True)
print(f'\n✅ Scaler fitted and saved!')
print(f'   Mean range: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]')
print(f'   Scale range: [{scaler.scale_.min():.4f}, {scaler.scale_.max():.4f}]')

# Show before/after normalization
X_train_scaled = transform(X_train, scaler)

print('\n📊 Before vs After Normalization (first 5 features, first 3 samples):')
before_after = pd.DataFrame({
    f'{feature_cols[i]}_raw': X_train[:3, i]
    for i in range(min(5, len(feature_cols)))
})
for i in range(min(5, len(feature_cols))):
    before_after[f'{feature_cols[i]}_scaled'] = X_train_scaled[:3, i]

display(before_after.style.format('{:.4f}'))

# Compute benign baseline for LLM analyzer
baseline = compute_benign_baseline(X_train, y_train, save=True)
print(f'\n✅ Benign baseline computed from {int((y_train == 0).sum())} benign samples')
print('   Top-5 features by mean value (benign baseline):')
sorted_baseline = sorted(baseline.items(), key=lambda x: -x[1]['mean'])
for name, stats in sorted_baseline[:5]:
    print(f'     {name}: mean={stats["mean"]:.4f}, std={stats["std"]:.4f}')

In [ ]:
# === Cell 6: Feature Extraction Visualization ===

print('='*70)
print('📊 FEATURE EXTRACTION VISUALIZATION')
print('='*70)

# Dark theme styling
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor': '#161B22',
    'axes.edgecolor': '#30363D',
    'axes.labelcolor': '#E6EDF3',
    'text.color': '#E6EDF3',
    'xtick.color': '#8B949E',
    'ytick.color': '#8B949E',
    'grid.color': '#21262D',
    'legend.facecolor': '#161B22',
    'legend.edgecolor': '#30363D',
})

SAFE_GREEN = '#00E59B'
ALERT_RED = '#FF4C6A'
ACCENT_PURPLE = '#7C4DFF'

if len(sample_pdfs) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('Feature Extraction: Benign vs Malicious PDF',
                 fontsize=16, fontweight='bold', y=1.02)
    
    for idx, pdf_path in enumerate(sorted(sample_pdfs)[:2]):
        feats = extract_features_dict(pdf_path)
        names = list(feats.keys())
        values = list(feats.values())
        
        color = SAFE_GREEN if 'benign' in pdf_path.name.lower() else ALERT_RED
        
        ax = axes[idx]
        bars = ax.barh(range(len(names)), values, color=color, alpha=0.8,
                       edgecolor='white', linewidth=0.3)
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names, fontsize=7)
        ax.set_xlabel('Feature Value', fontsize=10)
        ax.set_title(pdf_path.name, fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.2)
    
    plt.tight_layout()
    
    # Save figure
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / 'feature_extraction_comparison.png',
                dpi=150, bbox_inches='tight', facecolor='#0D1117')
    print(f'✅ Saved: {FIGURES_DIR / "feature_extraction_comparison.png"}')
    plt.show()
else:
    print('Need at least 2 sample PDFs for comparison visualization')

In [ ]:
# === Cell 7: End-to-End Pipeline Demo ===

print('='*70)
print('🚀 END-TO-END PIPELINE: pdf_to_vector()')
print('='*70)
print()

import time

for pdf_path in sorted(sample_pdfs):
    print(f'--- {pdf_path.name} ---')
    
    start = time.perf_counter()
    try:
        scaled_vec, raw_feats = pdf_to_vector(pdf_path, return_raw=True)
        elapsed_ms = (time.perf_counter() - start) * 1000
        
        print(f'  ⏱️  Extraction time: {elapsed_ms:.1f} ms')
        print(f'  📐 Scaled vector shape: {scaled_vec.shape}')
        print(f'  📊 Scaled range: [{scaled_vec.min():.4f}, {scaled_vec.max():.4f}]')
        print(f'  📊 Scaled mean: {scaled_vec.mean():.4f}')
        
        # Show top-5 features by absolute scaled value
        abs_sorted = np.argsort(np.abs(scaled_vec))[::-1][:5]
        print(f'  🔝 Top-5 features by |scaled value|:')
        for i in abs_sorted:
            print(f'      {FEATURE_COLUMNS[i]}: {scaled_vec[i]:.4f} '
                  f'(raw: {raw_feats.get(FEATURE_COLUMNS[i], 0):.2f})')
    except FileNotFoundError as e:
        print(f'  ⚠️ Scaler not fitted yet: {e}')
        print(f'  ℹ️ Run Cell 5 first to fit the scaler')
    print()

In [ ]:
# === Cell 8: Scaler Persistence & Validation ===

print('='*70)
print('💾 SCALER PERSISTENCE & VALIDATION')
print('='*70)
print()

# Test save/load roundtrip
try:
    loaded_scaler = load_scaler()
    print(f'✅ Scaler loaded successfully from {MODELS_DIR / "scaler.pkl"}')
    print(f'   n_features: {loaded_scaler.n_features_in_}')
    print(f'   mean vector: [{loaded_scaler.mean_[:3].round(4)}...]')
    print(f'   scale vector: [{loaded_scaler.scale_[:3].round(4)}...]')
    
    # Verify roundtrip accuracy
    test_input = np.random.rand(1, len(feature_cols))
    original_output = scaler.transform(test_input)
    loaded_output = loaded_scaler.transform(test_input)
    
    roundtrip_match = np.allclose(original_output, loaded_output)
    print(f'\n   Roundtrip accuracy test: {"✅ PASS" if roundtrip_match else "❌ FAIL"}')
    
except FileNotFoundError as e:
    print(f'⚠️ {e}')
    print('   Run Cell 5 first to fit and save the scaler.')

In [ ]:
# === Cell 9: Feature Engineering Pipeline Summary ===

print('='*70)
print('📋 FEATURE ENGINEERING PIPELINE — SUMMARY')
print('='*70)
print()

summary = {
    'Component': [
        'Structural Extractor',
        'Metadata Extractor',
        'Feature Vectorizer',
        'StandardScaler',
        'Benign Baseline',
    ],
    'Module': [
        'src/features/structural.py',
        'src/features/metadata.py',
        'src/features/vectorizer.py',
        'models/scaler.pkl',
        'models/benign_baseline.pkl',
    ],
    'Features': [
        '25 structural features',
        '12 metadata features',
        '37-dim combined vector',
        'Fitted on training data',
        'Mean/std per benign feature',
    ],
    'Status': ['✅ Complete', '✅ Complete', '✅ Complete', '✅ Saved', '✅ Saved'],
}

summary_df = pd.DataFrame(summary)
display(summary_df.style.set_properties(**{
    'background-color': '#161B22',
    'color': '#E6EDF3',
    'border-color': '#30363D'
}))

print()
print('📁 Files created in Phase 3:')
files = [
    ('src/features/structural.py', 'Structural feature extraction (25 features)'),
    ('src/features/metadata.py', 'Metadata feature extraction (12 features)'),
    ('src/features/vectorizer.py', 'Vectorization + scaler pipeline'),
    ('src/features/__init__.py', 'Package exports'),
    ('notebooks/03_feature_engineering.ipynb', 'This notebook'),
    ('models/scaler.pkl', 'Fitted StandardScaler (runtime)'),
    ('models/benign_baseline.pkl', 'Benign baseline stats (runtime)'),
    ('data/sample_pdfs/benign_sample.pdf', 'Test benign PDF'),
    ('data/sample_pdfs/malicious_sample.pdf', 'Test malicious PDF'),
]
for path, desc in files:
    full = project_root / path
    exists = '✅' if full.exists() else '⬜'
    size = f'{full.stat().st_size:,} bytes' if full.exists() else 'N/A'
    print(f'  {exists} {path} — {desc} ({size})')

print()
print('🔑 Key Design Decisions:')
print('  1. Raw byte regex scanning — no PDF rendering/execution (SEC-05)')
print('  2. 30-second timeout for large files (NFR-105)')
print('  3. Zeroed features on error — graceful degradation (NFR-201)')
print('  4. Column order enforced by config.FEATURE_COLUMNS — model compatibility')
print('  5. Benign baseline computed for LLM suspicious-feature detection')
print('  6. Scaler persisted for inference-time use by Streamlit app')
print()
print('🎯 Ready for Phase 4: Model Development!')